In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from collections import Counter

label_map = {
    "Low viral": 0,
    "Medium viral": 1,
    "High viral": 2
}

import os
from openai import OpenAI
import json
import pandas as pd

import numpy as np
# reverse map: 0->Low viral, 1->Medium viral, 2->High viral
reverse_label_map = {v: k for k, v in label_map.items()}


In [ ]:
DEEPSEEK_API_KEY='...'
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")


def get_response(msg_text,system_prompt,max_tokens=20):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": msg_text},
        ],
        temperature=0,          # more consistent scoring
        max_tokens=max_tokens,
        stream=False,
        response_format={
            'type': 'json_object'
        }
    )
    

    raw = response.choices[0].message.content.strip()
    return raw,response

    # Optional: validate JSON output
    
    

In [ ]:
def build_prompt_modifier(trial_num=0):
    
    file_preds=f'Out/trial{trial_num}/deepseek-chat_train_Text_labels.csv'
    left_on='ID'
    df_preds=pd.read_csv(file_preds)
    if 'ViralityLabel' in df_preds.columns:
        df_preds["label"]=df_preds["ViralityLabel"]
        df_preds=df_preds.drop(columns=['ViralityLabel'])        
    df_preds["y_pred"] = pd.to_numeric(df_preds["label"], errors="coerce")


    df_gt=pd.read_csv('train_df_train.csv')
    df_gt["y_true"] = df_gt["ViralityLabel"].map(label_map)

    df_eval = df_gt.merge(
        df_preds[["ID", "y_pred"]],
        left_on=left_on,
        right_on="ID",
        how="inner"
    )

    # Drop rows with missing labels
    df_eval = df_eval.dropna(subset=["y_true", "y_pred"])
    df_eval["y_true"] = df_eval["y_true"].astype(int)
    df_eval["y_pred"] = df_eval["y_pred"].astype(int)

    print("Matched samples:", len(df_eval), "/", len(df_gt))


    # let us find some messages where we failed


    df_eval['dist']=np.abs(df_eval["y_true"]-df_eval["y_pred"])
    df_mistakes=df_eval[df_eval['dist']!=0]
    df_mistakes=df_mistakes.sort_values(by=['dist'],ascending=False)

    build_query_string='''Above is the system prompt.
    Below are the cases where the model made a mistake in anticipating the virality label'''
    for index,row in df_mistakes.head().iterrows():    
        build_query_string+=f"\nText:{row['Text']},\nactual Label: {row['ViralityLabel']}"
        build_query_string+=f"\nassessed label={reverse_label_map[row['y_pred']]}\n"
    build_query_string+='''Based on the above mistakes, give me a new improved system prompt that reduces the mistakes.
Keep it short    
Make sure that the formatting is maintained.
Return ONLY valid JSON with this exact schema:
{
    "improved_system_prompt": ...
}
No extra keys. No explanation. No markdown.
'''    

    return build_query_string

    

In [ ]:
summarise_prompt='''
Summarise the above  improved system prompt.
Remove redundancies.
Return ONLY valid JSON with this exact schema:
{
    "improved_system_prompt": ...
}
No extra keys. No explanation. No markdown.
'''

In [ ]:
trial_num=19

with open(f'Out/trial{trial_num}/sys_prompt.txt','r') as f:
    system_prompt=f.read()

system_prompt=f'\n{system_prompt}\n'
build_query_string=build_prompt_modifier(trial_num=trial_num)


trials=30

for trial_num in range(trial_num+1,trials):
    #print(build_query_string)
    print(f'trial:{trial_num}')
    improved_system_prompt_raw,_=get_response(build_query_string,system_prompt,max_tokens=500)
    improved_system_prompt_raw,_=get_response(summarise_prompt,improved_system_prompt_raw,max_tokens=400)
    improved_system_prompt = json.loads(improved_system_prompt_raw)['improved_system_prompt']

    os.makedirs(f'Out/trial{trial_num}/',exist_ok=True)
    with open(f'Out/trial{trial_num}/sys_prompt.txt','w') as f:
        f.write(improved_system_prompt)

    df=pd.read_csv('train_df_train.csv')
    res=[]
    for idx,row in df.iterrows():
        #print(row['Text'])
        msg_text=row['Text']
        raw,_=get_response(msg_text,improved_system_prompt)
        result = json.loads(raw)
        #print(raw,result)        
        res.append({
            'ID':row['ID'],
            'label':int(result['virality_score'])
        })
        res_df=pd.DataFrame(res)
        res_df.to_csv(f'Out/trial{trial_num}/deepseek-chat_train_Text_labels.csv',index=False)    
        
    

    # end of recording results, evaluate now
    with open(f'Out/trial{trial_num}/sys_prompt.txt','r') as f:
        system_prompt=f.read()
    build_query_string=build_prompt_modifier(trial_num=trial_num)
    
    

In [ ]:
raw

In [ ]:
print(improved_system_prompt_raw)

In [ ]:
json.loads(improved_system_prompt_raw)

In [ ]:
improved_system_prompt_raw

In [ ]:
print(improved_system_prompt)

In [ ]:
resp,_=get_response(build_query_string,system_prompt,max_tokens=500)

In [ ]:
print(resp)

In [ ]:
build_query_string

In [ ]:
system_prompt

## Try openrouter

In [ ]:
import requests
import json




import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from collections import Counter

label_map = {
    "Low viral": 0,
    "Medium viral": 1,
    "High viral": 2
}

import os
import json
import pandas as pd

import numpy as np
# reverse map: 0->Low viral, 1->Medium viral, 2->High viral
reverse_label_map = {v: k for k, v in label_map.items()}


In [ ]:
openrouter_api_key='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'
def openrouter_gpt52_get_response(msg_text,system_prompt,max_tokens=200):
    response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {openrouter_api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "openai/gpt-5.2",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": msg_text},
        ],
        "response_format": {"type": "json_object"},
        "temperature": 0,
        "max_tokens": max_tokens, 
    },
    )

    result = response.json()
    #print(result)
    content = result["choices"][0]["message"]["content"]
    parsed = json.loads(content)


    #print(parsed["virality_score"])
    return parsed

In [ ]:
model_name='openai/gpt-5.2'

In [ ]:


system_prompt='''You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.
'''

trial_num=0
os.makedirs(f'Out/{model_name}/trial{trial_num}',exist_ok=True)
# save system_prompt
with open(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt','w') as f:
    f.write(system_prompt)


# now run on the training data
df_train=pd.read_csv('train_df_train.csv')
res=[]
for index,row in df_train.iterrows():
  ID=row['ID']
  text=row['Text']
  resp=openrouter_gpt52_get_response(text,system_prompt)
  resp=resp["virality_score"]
  res.append({
      'ID':row['ID'],
      'label':int(resp)     
  })  


In [ ]:
res_df=pd.DataFrame(res)
res_df.to_csv(f'Out/{model_name}/trial{trial_num}/train_Text_labels.csv',index=False)    


In [ ]:
def build_prompt_modifier(model_name=None,trial_num=0):
    
    file_preds=f'Out/{model_name}/trial{trial_num}/train_Text_labels.csv'
    left_on='ID'
    df_preds=pd.read_csv(file_preds)
    if 'ViralityLabel' in df_preds.columns:
        df_preds["label"]=df_preds["ViralityLabel"]
        df_preds=df_preds.drop(columns=['ViralityLabel'])        
    df_preds["y_pred"] = pd.to_numeric(df_preds["label"], errors="coerce")


    df_gt=pd.read_csv('train_df_train.csv')
    df_gt["y_true"] = df_gt["ViralityLabel"].map(label_map)

    df_eval = df_gt.merge(
        df_preds[["ID", "y_pred"]],
        left_on=left_on,
        right_on="ID",
        how="inner"
    )

    # Drop rows with missing labels
    df_eval = df_eval.dropna(subset=["y_true", "y_pred"])
    df_eval["y_true"] = df_eval["y_true"].astype(int)
    df_eval["y_pred"] = df_eval["y_pred"].astype(int)

    print("Matched samples:", len(df_eval), "/", len(df_gt))


    # let us find some messages where we failed


    df_eval['dist']=np.abs(df_eval["y_true"]-df_eval["y_pred"])
    df_mistakes=df_eval[df_eval['dist']!=0]
    df_mistakes=df_mistakes.sort_values(by=['dist'],ascending=False)

    build_query_string='''Above is the system prompt.
    Below are the cases where the model made a mistake in anticipating the virality label'''
    for index,row in df_mistakes.head().iterrows():    
        build_query_string+=f"\nText:{row['Text']},\nactual Label: {row['ViralityLabel']}"
        build_query_string+=f"\nassessed label={reverse_label_map[row['y_pred']]}\n"
    build_query_string+='''Based on the above mistakes, give me a new improved system prompt that reduces the mistakes.
Keep it short    
Make sure that the formatting is maintained.
Return ONLY valid JSON with this exact schema:
{
    "improved_system_prompt": ...
}
No extra keys. No explanation. No markdown.
'''    

    return build_query_string


In [ ]:
#build_prompt_modifier(model_name=model_name,trial_num=0)
summarise_prompt='''
Summarise the above  improved system prompt.
Remove redundancies.
Return ONLY valid JSON with this exact schema:
{
    "improved_system_prompt": ...
}
No extra keys. No explanation. No markdown.
'''

In [ ]:
trial_num=21

with open(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt','r') as f:
    system_prompt=f.read()

system_prompt=f'\n{system_prompt}\n'
build_query_string=build_prompt_modifier(trial_num=trial_num,model_name=model_name)


trials=30

for trial_num in range(trial_num+1,trials):
    #print(build_query_string)
    print(f'trial:{trial_num}')
    improved_system_prompt_raw=openrouter_gpt52_get_response(build_query_string,system_prompt,max_tokens=500)
    improved_system_prompt_raw=improved_system_prompt_raw['improved_system_prompt']
    improved_system_prompt_raw=openrouter_gpt52_get_response(summarise_prompt,improved_system_prompt_raw,max_tokens=400)
    improved_system_prompt = improved_system_prompt_raw['improved_system_prompt']

    os.makedirs(f'Out/{model_name}/trial{trial_num}/',exist_ok=True)
    with open(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt','w') as f:
        f.write(improved_system_prompt)

    df=pd.read_csv('train_df_train.csv')
    res=[]
    for idx,row in df.iterrows():
        ID=row['ID']
        text=row['Text']
        resp=openrouter_gpt52_get_response(text,system_prompt)
        resp=resp["virality_score"]
        res.append({
            'ID':row['ID'],
            'label':int(resp)     
        })          
    res_df=pd.DataFrame(res)
    res_df.to_csv(f'Out/{model_name}/trial{trial_num}/train_Text_labels.csv',index=False)
    


### Let us stick till 21 trials

## Now I go for Gemini

In [ ]:
import requests
import json
import os

import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from collections import Counter

label_map = {
    "Low viral": 0,
    "Medium viral": 1,
    "High viral": 2
}

import os
from openai import OpenAI
import json
import pandas as pd

import numpy as np
# reverse map: 0->Low viral, 1->Medium viral, 2->High viral
reverse_label_map = {v: k for k, v in label_map.items()}


model_name = "google/gemini-2.5-pro-preview"
openrouter_api_key='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'
def openrouter_gemini_get_response(msg_text, system_prompt, max_tokens=1000):
    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {openrouter_api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": model_name,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": msg_text},
                ],
                "response_format": {"type": "json_object"},
                "temperature": 0,
                "max_tokens": max_tokens,
            },
            timeout=30,
        )
        #print(response)

        response.raise_for_status()
        result = response.json()

        if "choices" not in result or not result["choices"]:
            return None

        content = result["choices"][0]["message"].get("content")
        if not content:
            return None

        return json.loads(content)

    except (requests.RequestException, json.JSONDecodeError, KeyError, TypeError):
        return None

In [11]:


system_prompt='''You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.
'''

parsed = openrouter_gemini_get_response(
    msg_text="OpenAI just dropped a new model.",
    system_prompt=system_prompt,
    max_tokens=700,
)

print(parsed)
print(parsed["virality_score"])

<Response [200]>
{'virality_score': 2}
2


In [16]:


system_prompt='''You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.
'''

trial_num=0
os.makedirs(f'Out/{model_name}/trial{trial_num}',exist_ok=True)
# save system_prompt
with open(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt','w') as f:
    f.write(system_prompt)


# now run on the training data
df_train=pd.read_csv('train_df_train.csv')
res=[]
for index,row in df_train.iterrows():
  ID=row['ID']
  text=row['Text']
  resp=openrouter_gemini_get_response(text,system_prompt,max_tokens=700)
  if not resp:
     continue
  resp=resp["virality_score"]
  res.append({
      'ID':row['ID'],
      'label':int(resp)     
  })  
res_df=pd.DataFrame(res)
res_df.to_csv(f'Out/{model_name}/trial{trial_num}/train_Text_labels.csv',index=False)


<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200

In [ ]:
model_name